# Uplift Churn — Interactive Dashboard

This notebook provides an **interactive exploration layer** on top of the trained
uplift models. It re-runs the full pipeline (takes ~2–3 min) and then renders:

| Section | What you get |
|---------|-------------|
| **1 — Model Performance** | Plotly Qini curves for all three models |
| **2 — Targeting Strategy** | Live ROI curve; sliders for contact cost & churn value |
| **3 — Individual Explainability** | Per-customer uplift score + SHAP waterfall |

Run **all cells top-to-bottom** once to populate the in-memory state, then
interact with the widgets in Sections 1–3.

---
## Setup

In [ ]:
# ── Colab: mount Drive and set repo path ───────────────────────────────────
import os, sys, pathlib

REPO_NAME  = "uplift-churn"            # folder name on your Drive root
DRIVE_ROOT = "/content/drive/MyDrive"

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_PATH = f"{DRIVE_ROOT}/{REPO_NAME}"
    if not os.path.isdir(REPO_PATH):
        raise FileNotFoundError(
            f"Folder not found: {REPO_PATH}\n"
            f"Upload the '{REPO_NAME}' folder to your Drive root and re-run."
        )
    sys.path.insert(0, REPO_PATH)
    os.chdir(REPO_PATH)

    DRIVE_OUT = pathlib.Path(REPO_PATH) / "outputs"
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    DRIVE_OUT = None

print('Working directory:', os.getcwd())
print('Python path entry:', sys.path[0])
if DRIVE_OUT:
    print('Outputs dir:', DRIVE_OUT)


In [ ]:
# ── Colab: install dependencies ─────────────────────────────────────────────
# Uses requirements-colab.txt (packages Colab doesn't pre-install).
# Full pinned versions are in requirements.txt for local dev.
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess, sys
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        raise RuntimeError('pip install failed — see output above')
    print('Dependencies installed.')
else:
    print('Local run — make sure your venv is activated and requirements installed.')


In [ ]:
# Drive is already mounted and DRIVE_OUT is set in the setup cell above.
print('DRIVE_OUT:', DRIVE_OUT)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 100

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML
import shap

from src import ingest, preprocess, models as m_module, evaluate

print('All imports OK.')

---
## Background Computation

The cell below runs the full pipeline (ingest → preprocess → train → evaluate).
This takes ~2–3 minutes. All interactive sections below depend on the state it creates.

In [ ]:
print('1/5  Downloading data...')
raw_df = ingest.load_raw()

print('2/5  Preprocessing...')
processed_df, (X_train, X_test, y_train, y_test, w_train, w_test) = preprocess.run(raw_df)

print('3/5  Training S-Learner, T-Learner, X-Learner...')
fitted_models = m_module.train_all(X_train, y_train, w_train)

print('4/5  Generating predictions...')
cate_preds = m_module.predict_all(fitted_models, X_test)

print('5/5  Computing AUUC scores and SHAP values...')
auuc_scores = {
    name: evaluate._auuc(cate, y_test.values, w_test.values)
    for name, cate in cate_preds.items()
}

best_name = max(auuc_scores, key=auuc_scores.get)
best_model = next(m for m in fitted_models if m.name == best_name)
shap_values = evaluate.compute_shap_values(best_model, X_train, X_test, n_background=200)

# Pre-compute Qini arrays for Plotly
qini_arrays = evaluate.compute_qini_arrays(cate_preds, y_test, w_test)

# Reset X_test index for consistent customer lookup
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
w_test = w_test.reset_index(drop=True)

print(f'\nDone. Best model: {best_name} (AUUC={auuc_scores[best_name]:.4f})')
print(f'Test set size: {len(X_test):,} customers | Features: {X_test.shape[1]}')


---
## Section 1 — Model Performance

**Qini curves** show how many *additional* churn events are prevented above random chance as
we contact progressively more customers, ranked by predicted uplift.

- Hover over each curve to read the exact value at any targeting fraction.
- The **black dashed line** is the random baseline (no uplift model).
- A higher, earlier-peaking curve means better targeted uplift.

In [ ]:
COLORS = {'S-Learner': '#2196F3', 'T-Learner': '#4CAF50', 'X-Learner': '#FF5722'}

fig = go.Figure()

for name, (xs, ys) in qini_arrays.items():
    auuc = auuc_scores[name]
    fig.add_trace(go.Scatter(
        x=xs, y=ys,
        mode='lines',
        name=f'{name}  (AUUC={auuc:.4f})',
        line=dict(color=COLORS.get(name, 'grey'), width=2.5),
        hovertemplate='Targeting: %{x:.1%}<br>Qini gain: %{y:.2f}<extra>' + name + '</extra>',
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 0],
    mode='lines',
    name='Random baseline',
    line=dict(color='black', dash='dash', width=1.5),
    hoverinfo='skip',
))

fig.update_layout(
    title=dict(text='Qini Curves — All Uplift Models', font=dict(size=16)),
    xaxis=dict(title='Proportion of customers targeted', tickformat='.0%'),
    yaxis=dict(title='Incremental Qini gain'),
    legend=dict(x=0.02, y=0.98),
    hovermode='x unified',
    template='plotly_white',
    height=420,
)
fig.show()

---
## Section 2 — Targeting Strategy

Adjust **contact cost** and **churn revenue value** using the sliders below.
The chart updates instantly, showing:

- The **Net Lift curve** for the best model across all targeting fractions
- A **vertical line** at the optimal cutoff (break-even CATE = cost ÷ revenue)
- The **maximum net lift** and the targeting fraction that achieves it

The optimal cutoff tells you: *"Target all customers whose predicted uplift exceeds this threshold — any more and you're spending more than you save."*

In [ ]:
cost_slider = widgets.FloatSlider(
    value=5.0, min=1.0, max=50.0, step=1.0,
    description='Contact cost ($):',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='480px'),
    continuous_update=False,
)
revenue_slider = widgets.FloatSlider(
    value=200.0, min=50.0, max=500.0, step=10.0,
    description='Churn value ($):',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='480px'),
    continuous_update=False,
)

roi_output = widgets.Output()

def _update_roi(change=None):
    contact_cost = cost_slider.value
    churn_revenue = revenue_slider.value
    n_total = len(X_test)

    # Build ROI curves for all models
    fig = go.Figure()
    opt_fracs = {}

    for name, cate in cate_preds.items():
        fracs, net_lifts = evaluate.compute_roi_curve(
            cate, n_total, contact_cost, churn_revenue, n_points=100
        )
        opt_fracs[name] = fracs[np.argmax(net_lifts)]
        fig.add_trace(go.Scatter(
            x=fracs, y=net_lifts,
            mode='lines',
            name=name,
            line=dict(color=COLORS.get(name, 'grey'), width=2),
            hovertemplate=(
                'Targeting: %{x:.1%}<br>Net Lift: $%{y:,.0f}'
                '<extra>' + name + '</extra>'
            ),
        ))

    # Vertical line at best model's optimal fraction
    best_frac = opt_fracs[best_name]
    best_cate = cate_preds[best_name]
    best_fracs, best_lifts = evaluate.compute_roi_curve(
        best_cate, n_total, contact_cost, churn_revenue, n_points=100
    )
    max_lift = float(best_lifts.max())
    y_range_min = float(min(0, best_lifts.min() * 1.1))
    y_range_max = max_lift * 1.15

    fig.add_vline(
        x=float(best_frac),
        line_dash='dot',
        line_color='crimson',
        annotation_text=f'Optimal: {best_frac:.1%}',
        annotation_position='top right',
        annotation_font_color='crimson',
    )
    fig.add_hline(y=0, line_dash='dash', line_color='grey', line_width=1)

    thresh = m_module.optimal_threshold(contact_cost, churn_revenue)
    fig.update_layout(
        title=dict(
            text=(
                f'Net Lift vs Targeting Fraction — '
                f'Contact=${contact_cost:.0f} | Churn value=${churn_revenue:.0f} | '
                f'Break-even CATE={thresh:.4f}'
            ),
            font=dict(size=13),
        ),
        xaxis=dict(title='Proportion of customers targeted', tickformat='.0%'),
        yaxis=dict(title='Net Lift ($)', tickformat='$,.0f'),
        legend=dict(x=0.75, y=0.98),
        hovermode='x unified',
        template='plotly_white',
        height=430,
    )

    with roi_output:
        roi_output.clear_output(wait=True)
        fig.show()
        print(
            f'  Best model ({best_name}) peaks at {best_frac:.1%} targeting '
            f'→ max net lift = ${max_lift:,.0f}'
        )

cost_slider.observe(_update_roi, names='value')
revenue_slider.observe(_update_roi, names='value')

display(widgets.VBox([cost_slider, revenue_slider]), roi_output)
_update_roi()  # initial render

---
## Section 3 — Individual Explainability

Select a customer from the test set using the dropdown. You will see:

1. Their **predicted uplift score** (CATE) from the best model
2. A **SHAP waterfall chart** showing which features pushed the prediction up or down
3. A **plain-English summary** of the top 3 drivers

The waterfall baseline is the mean CATE across all test customers. Features in red
increase this customer's predicted uplift; features in blue decrease it.

In [ ]:
# Feature names for readable explanations
FEATURE_LABELS = {
    'tenure': 'months with company',
    'MonthlyCharges': 'monthly bill ($)',
    'TotalCharges': 'total billed ($)',
    'SeniorCitizen': 'senior citizen',
    'gender': 'gender (0=F, 1=M)',
    'Contract': 'contract type',
    'InternetService': 'internet service',
    'charge_per_month': 'avg charge/month ($)',
    'is_long_tenure': 'long-tenure flag (≥24 mo)',
}

def _feature_label(name):
    return FEATURE_LABELS.get(name, name)


def _shap_to_sentence(sv_single, feature_names, top_n=3):
    """Convert a single-customer SHAP Explanation into plain English."""
    vals = sv_single.values
    data = sv_single.data
    abs_idx = np.argsort(-np.abs(vals))[:top_n]

    phrases = []
    for i in abs_idx:
        fname = _feature_label(feature_names[i])
        fval = data[i]
        direction = 'raises' if vals[i] > 0 else 'lowers'
        delta = abs(vals[i])
        if isinstance(fval, float):
            phrases.append(f"{fname} = {fval:.2f} → {direction} uplift by {delta:.3f}")
        else:
            phrases.append(f"{fname} = {fval} → {direction} uplift by {delta:.3f}")

    return (
        f"Top {top_n} drivers for this customer's predicted uplift:\n"
        + "\n".join(f"  {i+1}. {p}" for i, p in enumerate(phrases))
    )


# Customer selector — use integer position in reset X_test
n_test = len(X_test)
customer_dropdown = widgets.Dropdown(
    options=list(range(n_test)),
    value=0,
    description='Customer #:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='300px'),
)

explain_output = widgets.Output()


def _update_explanation(change=None):
    idx = customer_dropdown.value
    best_cate = cate_preds[best_name]
    cate_val = best_cate[idx]
    actual_churn = int(y_test.iloc[idx])
    was_treated = int(w_test.iloc[idx])

    with explain_output:
        explain_output.clear_output(wait=True)

        # ── Header ───────────────────────────────────────────────────────
        mean_cate = float(np.mean(best_cate))
        vs_avg = cate_val - mean_cate
        vs_str = f"+{vs_avg:.4f}" if vs_avg >= 0 else f"{vs_avg:.4f}"
        display(HTML(
            f"<div style='font-family:monospace; padding:8px; "
            f"background:#f8f9fa; border-left:4px solid #2196F3; margin-bottom:8px'>"
            f"<b>Customer #{idx}</b> &nbsp;|&nbsp; "
            f"Model: <b>{best_name}</b> &nbsp;|&nbsp; "
            f"Predicted CATE: <b style='color:#2196F3'>{cate_val:.4f}</b> "
            f"({vs_str} vs mean {mean_cate:.4f})<br>"
            f"Actual churn: <b>{'Yes' if actual_churn else 'No'}</b> &nbsp;|&nbsp; "
            f"Was treated: <b>{'Yes' if was_treated else 'No'}</b>"
            f"</div>"
        ))

        # ── SHAP waterfall ────────────────────────────────────────────────
        if shap_values is not None:
            fig_w, ax_w = plt.subplots(figsize=(8, 5))
            plt.sca(ax_w)
            shap.plots.waterfall(shap_values[idx], max_display=12, show=False)
            plt.title(f'SHAP Waterfall — Customer #{idx} ({best_name})', pad=10)
            plt.tight_layout()
            plt.show()

            # ── Plain-English explanation ─────────────────────────────────
            sentence = _shap_to_sentence(
                shap_values[idx],
                feature_names=list(X_test.columns),
            )
            display(HTML(
                f"<pre style='background:#fff3e0; padding:10px; "
                f"border-left:4px solid #FF9800; font-size:13px'>{sentence}</pre>"
            ))
        else:
            print('SHAP values not available — inner estimator could not be extracted.')


customer_dropdown.observe(_update_explanation, names='value')
display(customer_dropdown, explain_output)
_update_explanation()  # initial render